# 01. ONNX エクスポートと内部構造

`simple_demo.ipynb` の **Step 1（ONNX エクスポート）** を深く理解するためのノートブックです。

**前提ファイル**: なし（このノートブック内でモデルを訓練します）

In [ ]:
try:
    import google.colab, subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl", "onnx", "torch", "torchvision"])
except ImportError:
    pass

In [ ]:
from torch import nn

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 2, kernel_size=5, stride=2)
        self.conv2 = nn.Conv2d(2, 3, kernel_size=5, stride=2)
        self.relu  = nn.ReLU()
        self.d1    = nn.Linear(48, 48)
        self.d2    = nn.Linear(48, 10)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.view(-1, 48)
        x = self.relu(self.d1(x))
        return self.d2(x)

## モデルを訓練して ONNX にエクスポートする

まず簡単に訓練して、ONNX ファイルを作ります。

In [ ]:
import torch, torchvision, json, os
from torch import nn

transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.5,), (0.5,))
])
dataset    = torchvision.datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

model     = MyModel()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    for inputs, labels in dataloader:
        optimizer.zero_grad()
        criterion(model(inputs), labels).backward()
        optimizer.step()
    print(f"Epoch {epoch+1} 完了")

In [ ]:
model_path = 'network.onnx'
data_path  = 'input.json'
shape      = [1, 1, 28, 28]

model.eval()
x = torch.randn(1, *shape[1:], requires_grad=True)

torch.onnx.export(
    model, x, model_path,
    export_params=True, opset_version=10, do_constant_folding=True,
    input_names=['input'], output_names=['output'], dynamo=False
)
json.dump({"input_data": [x.detach().numpy().reshape(-1).tolist()]}, open(data_path, 'w'))
print(f"エクスポート完了: {model_path}  ({os.path.getsize(model_path):,} bytes)")

## ONNX とは何か

**ONNX（Open Neural Network Exchange）** = ニューラルネットワークを「どのフレームワークでも読める形式」で保存する規格。

EZKL が ONNX を使う理由：
```
PyTorch モデル
      ↓  torch.onnx.export
  network.onnx   ← EZKL が読み込む
      ↓  ezkl.compile_circuit
    ZK 回路
```

ONNX の中身は **有向非巡回グラフ（DAG）** で、ノード（操作）とエッジ（データの流れ）で構成されます。

## グラフ全体の確認

In [ ]:
import onnx

onnx_model = onnx.load(model_path)
graph      = onnx_model.graph

print(f"ONNX バージョン  : {onnx_model.ir_version}")
print(f"プロデューサー   : {onnx_model.producer_name} {onnx_model.producer_version}")
print(f"ノード総数       : {len(graph.node)}")
print(f"重み（initializer）: {len(graph.initializer)}")

## 入力・出力の shape を確認する

In [ ]:
print("=== グラフ入力 ===")
for inp in graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"  {inp.name}: shape={shape}")

print("\n=== グラフ出力 ===")
for out in graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"  {out.name}: shape={shape}")

## 全ノード（計算ステップ）を順番に確認する

各ノードは「op_type（何をするか）」「inputs（何を受け取るか）」「outputs（何を返すか）」で構成されます。

In [ ]:
for i, node in enumerate(graph.node):
    print(f"[{i:02d}] {node.op_type:10s} | 入力: {list(node.input)} | 出力: {list(node.output)}")

## op_type の意味

| op_type | PyTorch の対応 | 内容 |
|---|---|---|
| `Conv` | `nn.Conv2d` | 畳み込み（重みとの内積） |
| `Relu` | `nn.ReLU` | 0 以下を 0 にする |
| `Constant` | 定数 | reshape に使うサイズ定数 |
| `Reshape` | `.view()` | テンソルの形状変換（Flatten） |
| `Gemm` | `nn.Linear` | 行列積 + バイアス |

`Gemm` = General Matrix Multiply。`nn.Linear` は内部で `output = input @ weight.T + bias` を計算しており、これが Gemm に対応します。

## モデルの重み（initializer）を確認する

`graph.initializer` に Conv の kernel・bias、Linear の weight・bias が格納されています。

In [ ]:
total = 0
for init in graph.initializer:
    shape = list(init.dims)
    n = 1
    for d in shape: n *= d
    total += n
    print(f"  {init.name:35s}  shape={str(shape):20s}  {n:6,} params")
print(f"\n総パラメータ数: {total:,}")

## shape がどう変わるか追う

入力から出力まで、各操作で shape がどう変わるかを確認します。

In [ ]:
import torch

model.eval()
hooks = []
shapes = []

def hook(name):
    def fn(module, inp, out):
        shapes.append((name, tuple(inp[0].shape), tuple(out.shape)))
    return fn

for name, m in model.named_modules():
    if isinstance(m, (torch.nn.Conv2d, torch.nn.ReLU, torch.nn.Linear)):
        hooks.append(m.register_forward_hook(hook(name)))

with torch.no_grad():
    model(torch.randn(1, 1, 28, 28))

for h in hooks:
    h.remove()

print(f"{'レイヤー':10s}  {'入力 shape':25s}  {'出力 shape'}")
print("-" * 60)
for name, inp, out in shapes:
    print(f"{name:10s}  {str(inp):25s}  {str(out)}")

## ZK 回路との接続

**各 ONNX ノード = ZK 制約の塊** です。

```
[Conv]    → 掛け算・足し算の制約が大量に発生
[Relu]    → 出力 >= 0 の範囲証明制約
[Gemm]    → 行列積の乗算制約
[Reshape] → データ移動のみ（制約コストは低い）
```

パラメータ数が多いほど制約数が増え、証明生成に時間・メモリがかかります。

**Netron で可視化**: https://netron.app に `network.onnx` をドロップするとグラフを見られます。

---
次: `02_settings.ipynb` で visibility（公開・秘密・固定）を設定します。